# Ground AI Node — Project Kessler (AOID)

**Role:** Environmental & mathematical intelligence processor bridging the Phase 1
Physics Engine (ground radar / orbit propagator) and the downstream Edge Payload
(satellite flight computer / FPGA co-processor).

## Four-Phase Preprocessing Pipeline

| Phase | Description |
|---|---|
| **1 — Ephemeris & State Vector Init** | SGP4 propagation of primary & debris TLEs to TCA |
| **2 — Multi-Orbit Backstep** | `obs_start = TCA - 2 × T` for identical lighting geometry |
| **3 — Umbra Verification** | Skyfield solar shadow check at `obs_start`; override to `ABORT_VISION_USE_GROUND_RADAR` if in shadow |
| **4 — Refined CDM Output** | Enforced JSON contract with `observation_window_start_utc`, drag-adjusted covariance, and action decision |

## Communications Loop
- **Inbound:** Raw conjunction dict passed directly to `run_once()` (no REST/socket listener yet — Phase 1 integration is a future wiring step)
- **Outbound:** Refined CDM JSON → `shared/cdm_packet.json` + TCP broadcast on `127.0.0.1:8765`

In [44]:
# ── Cell 1: Imports & Config ──────────────────────────────────────────────
import json
import math
import os
import socket
import threading
import time
import logging
from datetime import datetime, timezone, timedelta
from pathlib import Path

import requests

# ── Optional: SGP4 ────────────────────────────────────────────────────────
try:
    from sgp4.api import Satrec, WGS84
    SGP4_AVAILABLE = True
except ImportError:
    SGP4_AVAILABLE = False

# ── Optional: Skyfield ───────────────────────────────────────────────────
try:
    from skyfield.api import load as sf_load
    SKYFIELD_AVAILABLE = True
except ImportError:
    SKYFIELD_AVAILABLE = False

# ── Optional: XGBoost ────────────────────────────────────────────────────
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
log = logging.getLogger("ground_ai_node")

# ── Config ───────────────────────────────────────────────────────────────
CONFIG = {
    # NOAA space-weather endpoints (live)
    "noaa_f107_url": "https://services.swpc.noaa.gov/products/summary/10cm-flux.json",
    "noaa_kp_url":   "https://services.swpc.noaa.gov/json/planetary_k_index_1m.json",
    "poll_interval_sec": 60,
    "http_timeout_sec":  5,
    # Output paths
    "shared_dir":      Path("shared"),
    "cdm_output_path": Path("shared/cdm_packet.json"),
    # TCP broadcast
    "tcp_host": "127.0.0.1",
    "tcp_port": 8765,
    # XGBoost model (optional — falls back to heuristic if not present)
    "model_path": Path("models/drag_xgb_model.json"),
    # Conservative quiet-sun defaults (PRD §5)
    "default_f107": 150.0,
    "default_kp":   3.0,
    # Ballistic coefficient defaults (kg/m²)
    "default_bc_satellite": 100.0,
    "default_bc_debris":     30.0,
    # Action decision threshold
    "optical_confirmation_multiplier_threshold": 1.15,
    # Skyfield ephemeris cache dir
    "skyfield_data_dir": Path("skyfield_data"),
    # Earth radius for cylindrical umbra approximation
    "earth_radius_km": 6371.0,
}
CONFIG["shared_dir"].mkdir(parents=True, exist_ok=True)
CONFIG["skyfield_data_dir"].mkdir(parents=True, exist_ok=True)

print(f"SGP4={SGP4_AVAILABLE}  Skyfield={SKYFIELD_AVAILABLE}  XGBoost={XGBOOST_AVAILABLE}")


SGP4=True  Skyfield=True  XGBoost=True


In [45]:
# ── Cell 2: EphemerisProcessor — Phases 1, 2, 3 ──────────────────────────


class EphemerisProcessor:
    """Executes the three mathematical preprocessing phases.

    Phase 1 — SGP4 propagation
        Ingests TLE strings for the primary satellite and debris object,
        propagates both to the nominal TCA to establish baseline ECI
        (Earth-Centred Inertial) state vectors in km.

    Phase 2 — Multi-orbit backstep (observation scheduling)
        Extracts the primary mean motion n0 (rad/min) from the TLE and
        computes the exact orbital period T = 2*pi / n0 (min).
        Steps back exactly 2*T from TCA to find obs_start — guaranteeing
        identical Earth-Sun lighting geometry with ~3 h lead time.

    Phase 3 — Umbra (eclipse) verification
        Uses Skyfield + DE421 JPL ephemeris to get the Sun ECI vector at
        obs_start. Projects the satellite position onto the Earth-Sun axis;
        if the satellite falls inside Earth shadow cylinder
        (proj < 0 AND perp_dist < R_Earth) optical tracking is blind.
        Safety override: action -> ABORT_VISION_USE_GROUND_RADAR.
    """

    # Default TLEs for Kessler_Sat_1 (ISS-derived LEO ~400 km)
    DEFAULT_PRIMARY_TLE = (
        "1 25544U 98067A   26250.54791667  .00002182  00000-0  48260-4 0  9991",
        "2 25544  51.6416  49.1234 0005770  81.2000 278.9700 15.50377932123456",
    )
    # Default TLEs for a generic debris fragment (slightly higher orbit)
    DEFAULT_DEBRIS_TLE = (
        "1 45678U 20001A   26250.54791667  .00001500  00000-0  35000-4 0  9991",
        "2 45678  51.6500  50.0000 0012000  85.0000 275.0000 15.48000000 98765",
    )

    def __init__(self, config: dict):
        self.config = config
        self._ts  = None
        self._eph = None

    # ── Skyfield lazy-loading ──────────────────────────────────────────
    def _load_skyfield(self):
        """Load (or reuse) Skyfield timescale + DE421 ephemeris."""
        if self._ts is not None:
            return self._ts, self._eph
        try:
            self._ts  = sf_load.timescale()
            self._eph = sf_load("de421.bsp")
        except Exception:
            from skyfield.api import Loader
            ldr = Loader(str(self.config["skyfield_data_dir"]))
            self._ts  = ldr.timescale()
            self._eph = ldr("de421.bsp")
        return self._ts, self._eph

    # ── Phase 1 ───────────────────────────────────────────────────────
    @staticmethod
    def _datetime_to_jd(dt: datetime):
        """Convert a UTC datetime to (jd_whole, fraction) for sgp4."""
        y, mo, d = dt.year, dt.month, dt.day
        jd = (367 * y
              - int(7 * (y + int((mo + 9) / 12)) / 4)
              + int(275 * mo / 9)
              + d + 1721013.5)
        fr = (dt.hour * 3600 + dt.minute * 60 + dt.second
              + dt.microsecond / 1e6) / 86400.0
        return jd, fr

    def propagate_to_tca(
        self,
        tca_dt: datetime,
        primary_tle: tuple = None,
        debris_tle: tuple = None,
    ) -> dict:
        """Propagate primary and debris to TCA via SGP4.

        Returns ECI position vectors (km) and the primary mean motion.
        Falls back to placeholder vectors if sgp4 is unavailable.
        """
        primary_tle = primary_tle or self.DEFAULT_PRIMARY_TLE
        debris_tle  = debris_tle  or self.DEFAULT_DEBRIS_TLE

        if not SGP4_AVAILABLE:
            log.warning("sgp4 not available — using placeholder state vectors")
            return {
                "primary_eci_km": [6771.0, 0.0, 0.0],
                "debris_eci_km":  [6772.2, 0.0, 0.0],
                "primary_mean_motion_rad_per_min": 2.0 * math.pi / 92.4,
                "sgp4_used": False,
            }

        sat_p = Satrec.twoline2rv(primary_tle[0], primary_tle[1])
        sat_d = Satrec.twoline2rv(debris_tle[0],  debris_tle[1])

        jd, fr = self._datetime_to_jd(tca_dt)
        e1, r1, _ = sat_p.sgp4(jd, fr)
        e2, r2, _ = sat_d.sgp4(jd, fr)

        p_eci = [round(c, 4) for c in r1] if e1 == 0 else [6771.0, 0.0, 0.0]
        d_eci = [round(c, 4) for c in r2] if e2 == 0 else [6772.2, 0.0, 0.0]

        log.info(
            "Phase 1 | primary_eci=%s km  debris_eci=%s km  n0=%.6f rad/min",
            p_eci, d_eci, sat_p.no_kozai,
        )
        return {
            "primary_eci_km": p_eci,
            "debris_eci_km":  d_eci,
            "primary_mean_motion_rad_per_min": sat_p.no_kozai,
            "sgp4_used": True,
        }

    # ── Phase 2 ───────────────────────────────────────────────────────
    def compute_observation_window(
        self,
        tca_dt: datetime,
        mean_motion_rad_per_min: float,
    ) -> dict:
        """Compute obs_start = TCA - 2*T where T = 2*pi / n0."""
        period_min   = 2.0 * math.pi / mean_motion_rad_per_min
        backstep_min = 2.0 * period_min
        obs_start    = tca_dt - timedelta(minutes=backstep_min)
        log.info(
            "Phase 2 | T=%.2f min  backstep=%.2f min  obs_start=%s",
            period_min, backstep_min,
            obs_start.strftime("%Y-%m-%dT%H:%M:%SZ"),
        )
        return {
            "orbital_period_min": round(period_min, 4),
            "backstep_min":       round(backstep_min, 4),
            "obs_start_utc":      obs_start,
        }

    # ── Phase 3 ───────────────────────────────────────────────────────
    def check_umbra(self, obs_start_utc: datetime, satellite_eci_km: list) -> dict:
        """Cylindrical umbra check at obs_start_utc.

        Geometry:
            sun_hat  = Sun_ECI / |Sun_ECI|
            proj_km  = dot(sat_eci, sun_hat)   (positive = sunward side)
            perp_km  = |sat_eci - proj * sun_hat|   (from Earth-Sun axis)
            in_umbra = (proj_km < 0) AND (perp_km < R_Earth)
        """
        if not SKYFIELD_AVAILABLE:
            log.warning("skyfield not available — assuming NOT in umbra")
            return {
                "is_in_umbra": False, "skyfield_used": False,
                "sun_eci_km": None, "perp_dist_km": None, "proj_km": None,
            }

        try:
            ts, eph = self._load_skyfield()
            if obs_start_utc.tzinfo is None:
                obs_start_utc = obs_start_utc.replace(tzinfo=timezone.utc)
            t = ts.from_datetime(obs_start_utc)

            sun_km = (eph["sun"] - eph["earth"]).at(t).position.km

            sat       = satellite_eci_km
            sun_norm  = math.sqrt(sum(float(c)**2 for c in sun_km))
            sun_hat   = [float(c) / sun_norm for c in sun_km]
            proj_km   = sum(sat[i] * sun_hat[i] for i in range(3))
            perp_vec  = [sat[i] - proj_km * sun_hat[i] for i in range(3)]
            perp_km   = math.sqrt(sum(c**2 for c in perp_vec))

            R_earth    = self.config["earth_radius_km"]
            in_umbra   = (proj_km < 0.0) and (perp_km < R_earth)

            log.info(
                "Phase 3 | proj=%.1f km  perp=%.1f km  R_earth=%.1f km  umbra=%s",
                proj_km, perp_km, R_earth, in_umbra,
            )
            return {
                "is_in_umbra":  in_umbra,
                "skyfield_used": True,
                "sun_eci_km":   [round(float(c), 2) for c in sun_km],
                "perp_dist_km": round(perp_km, 2),
                "proj_km":      round(proj_km, 2),
            }
        except Exception as exc:
            log.warning("Umbra check failed (%s) — defaulting is_in_umbra=False", exc)
            return {
                "is_in_umbra": False, "skyfield_used": False,
                "sun_eci_km": None, "perp_dist_km": None, "proj_km": None,
            }

    # ── Orchestrate all three phases ───────────────────────────────────
    def run(
        self,
        tca_dt: datetime,
        primary_tle: tuple = None,
        debris_tle: tuple = None,
    ) -> dict:
        """Execute Phases 1-3; return combined preprocessing dict."""
        ephem   = self.propagate_to_tca(tca_dt, primary_tle, debris_tle)
        obs_win = self.compute_observation_window(
            tca_dt, ephem["primary_mean_motion_rad_per_min"]
        )
        umbra = self.check_umbra(obs_win["obs_start_utc"], ephem["primary_eci_km"])
        return {
            "primary_eci_km":      ephem["primary_eci_km"],
            "debris_eci_km":       ephem["debris_eci_km"],
            "mean_motion_rad_min": ephem["primary_mean_motion_rad_per_min"],
            "sgp4_used":           ephem["sgp4_used"],
            "orbital_period_min":  obs_win["orbital_period_min"],
            "obs_start_utc":       obs_win["obs_start_utc"],
            **umbra,
        }


print("EphemerisProcessor defined OK")


EphemerisProcessor defined OK


In [46]:
# ── Cell 3: DataIngestor ─────────────────────────────────────────────────


class DataIngestor:
    """Polls NOAA space-weather APIs for F10.7 and Kp.

    Falls back to last-known-good cache, then to conservative quiet-sun
    defaults.  Never raises — a bad poll must never kill the service.
    """

    def __init__(self, config: dict):
        self.config = config
        self._cache = {"f107": None, "kp": None}
        self._stale = False
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        }

    def _get_json(self, url: str):
        resp = requests.get(url, headers=self.headers,timeout=10)
        resp.raise_for_status()
        return resp.json()

    def fetch_f107(self) -> float:
        try:
            data  = self._get_json(self.config["noaa_f107_url"])
            value = float(data[-1]["flux"])
            self._cache["f107"] = value
            return value
        except Exception as exc:
            log.warning("F10.7 fetch failed (%s); falling back", exc)
            return self._fallback("f107")

    def fetch_kp(self) -> float:
        try:
            data   = self._get_json(self.config["noaa_kp_url"])
            latest = data[-1]
            raw    = latest.get("estimated_kp")
            value  = float(raw) if raw is not None else float(latest["kp_index"])
            self._cache["kp"] = value
            return value
        except Exception as exc:
            log.warning("Kp fetch failed (%s); falling back", exc)
            return self._fallback("kp")

    def _fallback(self, key: str) -> float:
        if self._cache[key] is not None:
            self._stale = True
            log.info("Using cached %s = %s (stale_data=True)", key, self._cache[key])
            return self._cache[key]
        default = self.config[f"default_{key}"]
        self._stale = True
        log.info("No cache for %s; using default = %s", key, default)
        return default

    def poll(self) -> dict:
        self._stale = False
        f107 = self.fetch_f107()
        kp   = self.fetch_kp()
        return {"f107": f107, "kp": kp, "stale_data": self._stale}


print("DataIngestor defined OK")


DataIngestor defined OK


In [47]:
# ── Cell 4: FeatureBuilder ───────────────────────────────────────────────


class FeatureBuilder:
    """Assembles the fixed 4-feature vector: [f107, kp, BC_sat, BC_debris]."""

    def __init__(self, config: dict):
        self.config = config

    def build(self, weather: dict, conjunction: dict) -> dict:
        bc_sat    = conjunction.get("bc_satellite")
        bc_debris = conjunction.get("bc_debris")
        bc_defaulted = False

        if bc_sat is None:
            bc_sat = self.config["default_bc_satellite"]
            bc_defaulted = True
        if bc_debris is None:
            bc_debris = self.config["default_bc_debris"]
            bc_defaulted = True

        if bc_defaulted:
            log.info("bc_defaulted=True (one or both ballistic coefficients missing)")

        return {
            "f107":         weather["f107"],
            "kp":           weather["kp"],
            "bc_satellite": bc_sat,
            "bc_debris":    bc_debris,
            "bc_defaulted": bc_defaulted,
            "stale_data":   weather.get("stale_data", False),
        }

    @staticmethod
    def to_vector(features: dict) -> list:
        return [
            features["f107"], features["kp"],
            features["bc_satellite"], features["bc_debris"],
        ]


print("FeatureBuilder defined OK")


FeatureBuilder defined OK


In [48]:
# ── Cell 5: DragPredictor ────────────────────────────────────────────────


class DragPredictor:
    """XGBoost regressor (if model exists) or physically-motivated heuristic.

    Baseline: f107=150, kp=3 -> multiplier = 1.0
    """

    def __init__(self, config: dict):
        self.config = config
        self.model  = None
        if XGBOOST_AVAILABLE and config["model_path"].exists():
            try:
                self.model = xgb.XGBRegressor()
                self.model.load_model(str(config["model_path"]))
                log.info("Loaded XGBoost model from %s", config["model_path"])
            except Exception as exc:
                log.warning("Model load failed (%s); using _mock_inference", exc)
                self.model = None
        else:
            log.info("No trained model found — using _mock_inference heuristic")

    def _mock_inference(self, features: dict):
        f107      = features["f107"]
        kp        = features["kp"]
        bc_debris = features["bc_debris"]

        f107_term = (f107 - 150.0) / 150.0
        kp_term   = (kp   -   3.0) /   9.0

        dm = 1.0 + 0.6 * f107_term + 0.4 * kp_term
        dm = max(0.5, min(3.0, dm))

        bc_sensitivity = 30.0 / max(bc_debris, 1.0)
        base_sigma     = 50.0 * bc_sensitivity * dm

        return dm, [
            round(base_sigma * 1.0, 2),   # sigma_Radial   (m)
            round(base_sigma * 3.0, 2),   # sigma_In-track (m) — drag dominates
            round(base_sigma * 1.0, 2),   # sigma_Cross-track (m)
        ]

    def predict(self, features: dict):
        if self.model is not None:
            try:
                vec = [FeatureBuilder.to_vector(features)]
                dm  = float(self.model.predict(vec)[0])
                _, cov = self._mock_inference(features)
                return dm, cov
            except Exception as exc:
                log.warning("Model inference failed (%s); falling back", exc)
        return self._mock_inference(features)


print("DragPredictor defined OK")


DragPredictor defined OK


In [49]:
# ── Cell 6: CDMBuilder — Phase 4 Enforced JSON Output Contract ───────────


class CDMBuilder:
    """Merges conjunction geometry + ephemeris + AI drag into Refined CDM.

    Action decision logic (first match wins):
      1. is_in_umbra == True               -> ABORT_VISION_USE_GROUND_RADAR
      2. |dm - 1.0| >= threshold           -> RECOMMEND_OPTICAL_CONFIRMATION
      3. miss_dist <= 3 * sigma_in-track   -> RECOMMEND_OPTICAL_CONFIRMATION
      4. otherwise                         -> NO_ACTION
    """

    def __init__(self, config: dict):
        self.config = config

    def _decide_action(
        self,
        drag_multiplier: float,
        covariance: list,
        miss_distance_km: float,
        is_in_umbra: bool = False,
    ) -> str:
        if is_in_umbra:
            return "ABORT_VISION_USE_GROUND_RADAR"

        threshold = self.config["optical_confirmation_multiplier_threshold"]
        drag_triggered = abs(drag_multiplier - 1.0) >= (threshold - 1.0)

        sigma_in_track_km = covariance[1] / 1000.0
        uncertainty_triggered = miss_distance_km <= (3.0 * sigma_in_track_km)

        if drag_triggered or uncertainty_triggered:
            return "RECOMMEND_OPTICAL_CONFIRMATION"
        return "NO_ACTION"

    def build(
        self,
        conjunction: dict,
        features: dict,
        drag_multiplier: float,
        covariance: list,
        preprocessing: dict = None,
    ) -> dict:
        """Build the Phase 4 enforced JSON output contract packet."""
        preprocessing = preprocessing or {}
        miss_dist   = conjunction.get("miss_distance_km", 999.0)
        is_in_umbra = preprocessing.get("is_in_umbra", False)

        action = self._decide_action(drag_multiplier, covariance, miss_dist, is_in_umbra)

        obs_start = preprocessing.get("obs_start_utc")
        if isinstance(obs_start, datetime):
            obs_start_str = obs_start.strftime("%Y-%m-%dT%H:%M:%SZ")
        else:
            obs_start_str = obs_start

        # ── Phase 4: Enforced JSON Output Contract ────────────────────
        return {
            "header": {
                "type":          "REFINED_CDM",
                "timestamp_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            },
            "conjunction_data": {
                "primary_asset":               conjunction["primary_asset"],
                "secondary_asset":             conjunction["secondary_asset"],
                "time_of_closest_approach":    conjunction["time_of_closest_approach"],
                "observation_window_start_utc": obs_start_str,
                "miss_distance_km":            miss_dist,
            },
            "ai_drag_prediction": {
                "f107_flux":                  features["f107"],
                "kp_index":                   features["kp"],
                "drag_multiplier":            round(drag_multiplier, 4),
                "ellipsoid_covariance_matrix": covariance,
            },
            "action": action,
            "_audit": {
                "stale_data":         features.get("stale_data",   False),
                "bc_defaulted":       features.get("bc_defaulted", False),
                "sgp4_used":          preprocessing.get("sgp4_used",     False),
                "skyfield_used":      preprocessing.get("skyfield_used",  False),
                "is_in_umbra":        is_in_umbra,
                "orbital_period_min": preprocessing.get("orbital_period_min"),
            },
        }


print("CDMBuilder defined OK")


CDMBuilder defined OK


In [50]:
# ── Cell 7: Publisher ────────────────────────────────────────────────────


class Publisher:
    """Writes the Refined CDM atomically to disk + TCP broadcast.

    Both paths share the same in-memory packet so they can never disagree.
    Never blocks/crashes the main loop on a bad client connection.
    """

    def __init__(self, config: dict):
        self.config         = config
        self._clients       = []
        self._clients_lock  = threading.Lock()
        self._server_thread = None
        self._server_socket = None
        self._running       = False

    def start_tcp_server(self):
        self._running = True
        self._server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self._server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        self._server_socket.bind((self.config["tcp_host"], self.config["tcp_port"]))
        self._server_socket.listen(5)
        self._server_socket.settimeout(1.0)

        def accept_loop():
            log.info(
                "TCP publisher listening on %s:%s",
                self.config["tcp_host"], self.config["tcp_port"],
            )
            while self._running:
                try:
                    conn, addr = self._server_socket.accept()
                    with self._clients_lock:
                        self._clients.append(conn)
                    log.info("TCP client connected: %s", addr)
                except socket.timeout:
                    continue
                except OSError:
                    break

        self._server_thread = threading.Thread(target=accept_loop, daemon=True)
        self._server_thread.start()

    def stop_tcp_server(self):
        self._running = False
        if self._server_socket:
            self._server_socket.close()
        with self._clients_lock:
            for c in self._clients:
                try:
                    c.close()
                except OSError:
                    pass
            self._clients = []

    def publish(self, packet: dict, out_path: Path = None) -> dict:
        """Write atomically to out_path (or default) + TCP broadcast."""
        payload  = json.dumps(packet, indent=2)
        out_path = out_path or self.config["cdm_output_path"]
        tmp_path = out_path.with_suffix(".tmp")
        try:
            tmp_path.write_text(payload)
            os.replace(tmp_path, out_path)
        except Exception as exc:
            log.error("File publish failed (%s)", exc)

        line = (json.dumps(packet) + "\n").encode("utf-8")
        with self._clients_lock:
            alive = []
            for conn in self._clients:
                try:
                    conn.sendall(line)
                    alive.append(conn)
                except OSError:
                    try:
                        conn.close()
                    except OSError:
                        pass
            self._clients = alive
        return packet


print("Publisher defined OK")


Publisher defined OK


In [51]:
# ── Cell 8: GroundAINode — Orchestrator ──────────────────────────────────


class GroundAINode:
    """Top-level orchestrator wiring all pipeline stages:

        EphemerisProcessor  (Phases 1-3: SGP4 + obs-window + umbra check)
          |
        DataIngestor        (NOAA F10.7 + Kp space weather)
          |
        FeatureBuilder      (4-feature vector assembly)
          |
        DragPredictor       (XGBoost model or physically-motivated heuristic)
          |
        CDMBuilder          (Phase 4: Refined CDM JSON contract)
          |
        Publisher           (atomic file write + TCP newline-JSON broadcast)

    Usage:
        node.run_once(conjunction)           — one synchronous cycle
        node.start(conjunction, interval=10) — starts background loop + TCP
        node.stop()                          — graceful shutdown
    """

    def __init__(self, config: dict):
        self.config          = config
        self.ephemeris       = EphemerisProcessor(config)
        self.ingestor        = DataIngestor(config)
        self.feature_builder = FeatureBuilder(config)
        self.predictor       = DragPredictor(config)
        self.cdm_builder     = CDMBuilder(config)
        self.publisher       = Publisher(config)
        self._thread         = None
        self._running        = False
        self.last_packet     = None

    def run_once(
        self,
        conjunction: dict,
        out_path: Path = None,
        primary_tle: tuple = None,
        debris_tle: tuple = None,
    ) -> dict:
        """Execute one full pipeline cycle synchronously.

        Args:
            conjunction:  Raw CDM dict from Phase 1 Physics Engine.
                          Required keys: primary_asset, secondary_asset,
                          time_of_closest_approach, miss_distance_km.
                          Optional: bc_satellite, bc_debris.
            out_path:     Override output file path (useful for testing).
            primary_tle:  (line1, line2) TLE strings for primary satellite.
            debris_tle:   (line1, line2) TLE strings for debris object.

        Returns:
            Refined CDM dict (Phase 4 enforced JSON contract).
        """
        tca_str = conjunction["time_of_closest_approach"]
        tca_dt  = datetime.fromisoformat(
            tca_str.replace("Z", "+00:00")
        ).replace(tzinfo=timezone.utc)

        preprocessing = self.ephemeris.run(tca_dt, primary_tle, debris_tle)
        weather       = self.ingestor.poll()
        features      = self.feature_builder.build(weather, conjunction)
        dm, covariance = self.predictor.predict(features)

        packet = self.cdm_builder.build(
            conjunction, features, dm, covariance, preprocessing
        )
        self.publisher.publish(packet, out_path=out_path)
        self.last_packet = packet
        return packet

    def start(
        self,
        conjunction: dict,
        interval_sec: int = None,
        primary_tle: tuple = None,
        debris_tle: tuple = None,
    ):
        """Start TCP server + background polling loop."""
        interval_sec = interval_sec or self.config["poll_interval_sec"]
        self.publisher.start_tcp_server()
        self._running = True

        def loop():
            while self._running:
                try:
                    pkt = self.run_once(
                        conjunction,
                        primary_tle=primary_tle,
                        debris_tle=debris_tle,
                    )
                    log.info(
                        "Published CDM | drag=%.3f | action=%s",
                        pkt["ai_drag_prediction"]["drag_multiplier"],
                        pkt["action"],
                    )
                except Exception as exc:
                    log.error("Cycle failed (%s) — continuing", exc)
                time.sleep(interval_sec)

        self._thread = threading.Thread(target=loop, daemon=True)
        self._thread.start()
        log.info("GroundAINode started (interval=%ss)", interval_sec)

    def stop(self):
        """Graceful shutdown."""
        self._running = False
        self.publisher.stop_tcp_server()
        log.info("GroundAINode stopped")


print("GroundAINode defined OK")


GroundAINode defined OK


## Demo Run

Executes one full pipeline cycle with sample conjunction geometry.
Demonstrates the **Phase 4 enforced JSON contract** including
`observation_window_start_utc` and the Phase 3 umbra safety flag.


In [52]:
# ── Cell 9: Demo run (synchronous) ──────────────────────────────────────

SAMPLE_CONJUNCTION = {
    "primary_asset":            "Kessler_Sat_1",
    "secondary_asset":          "Debris_Obj_8492",
    "time_of_closest_approach": "2026-09-08T14:32:00Z",
    "miss_distance_km":         1.2,
    # bc_satellite and bc_debris omitted to exercise bc_defaulted path
    "bc_satellite": None,
    "bc_debris":    None,
}

node   = GroundAINode(CONFIG)
packet = node.run_once(SAMPLE_CONJUNCTION)
print(json.dumps(packet, indent=2))


2026-09-07 22:52:29,092 [INFO] ground_ai_node: No trained model found — using _mock_inference heuristic
2026-09-07 22:52:29,094 [INFO] ground_ai_node: Phase 1 | primary_eci=[-5696.1949, -2317.2068, 2876.2932] km  debris_eci=[-5570.8007, -1559.4711, 3557.5147] km  n0=0.067648 rad/min
2026-09-07 22:52:29,095 [INFO] ground_ai_node: Phase 2 | T=92.88 min  backstep=185.76 min  obs_start=2026-09-08T11:26:14Z
2026-09-07 22:52:29,102 [INFO] ground_ai_node: Phase 3 | proj=5267.6 km  perp=4282.7 km  R_earth=6371.0 km  umbra=False
2026-09-07 22:52:30,911 [INFO] ground_ai_node: bc_defaulted=True (one or both ballistic coefficients missing)


{
  "header": {
    "type": "REFINED_CDM",
    "timestamp_utc": "2026-09-07T17:22:30Z"
  },
  "conjunction_data": {
    "primary_asset": "Kessler_Sat_1",
    "secondary_asset": "Debris_Obj_8492",
    "time_of_closest_approach": "2026-09-08T14:32:00Z",
    "observation_window_start_utc": "2026-09-08T11:26:14Z",
    "miss_distance_km": 1.2
  },
  "ai_drag_prediction": {
    "f107_flux": 110.0,
    "kp_index": 3.33,
    "drag_multiplier": 0.8547,
    "ellipsoid_covariance_matrix": [
      42.73,
      128.2,
      42.73
    ]
  },
  "action": "NO_ACTION",
  "_audit": {
    "stale_data": false,
    "bc_defaulted": true,
    "sgp4_used": true,
    "skyfield_used": true,
    "is_in_umbra": false,
    "orbital_period_min": 92.8806
  }
}


In [53]:
# ── Cell 10: Confirm file-based transport ───────────────────────────────
print(CONFIG["cdm_output_path"].read_text())


{
  "header": {
    "type": "REFINED_CDM",
    "timestamp_utc": "2026-09-07T17:22:30Z"
  },
  "conjunction_data": {
    "primary_asset": "Kessler_Sat_1",
    "secondary_asset": "Debris_Obj_8492",
    "time_of_closest_approach": "2026-09-08T14:32:00Z",
    "observation_window_start_utc": "2026-09-08T11:26:14Z",
    "miss_distance_km": 1.2
  },
  "ai_drag_prediction": {
    "f107_flux": 110.0,
    "kp_index": 3.33,
    "drag_multiplier": 0.8547,
    "ellipsoid_covariance_matrix": [
      42.73,
      128.2,
      42.73
    ]
  },
  "action": "NO_ACTION",
  "_audit": {
    "stale_data": false,
    "bc_defaulted": true,
    "sgp4_used": true,
    "skyfield_used": true,
    "is_in_umbra": false,
    "orbital_period_min": 92.8806
  }
}


### Start the live poll → predict → publish loop (optional)

Starts the TCP server + background thread in this kernel.
Basilisk / Streamlit / edge nodes connect to `127.0.0.1:8765` for
newline-delimited JSON, or simply poll `shared/cdm_packet.json`.


In [ ]:
# ── Cell 11: Start background loop ──────────────────────────────────────
node.start(SAMPLE_CONJUNCTION, interval_sec=10)


In [ ]:
# ── Cell 12: Stop background loop ───────────────────────────────────────
node.stop()


### Minimal TCP client (for local testing)

```python
import socket, json

s   = socket.create_connection((CONFIG["tcp_host"], CONFIG["tcp_port"]), timeout=5)
buf = s.recv(65536).decode()
print(json.loads(buf.strip().split("\n")[0]))
s.close()
```

### Swapping in a trained model

Save an XGBoost model to `models/drag_xgb_model.json` (matching `CONFIG["model_path"]`)
and re-instantiate `DragPredictor`. No changes to ingestion, ephemeris, or publishing
are needed. Until that file exists, every cycle uses `_mock_inference()` transparently.

### TLE Updates

Pass `primary_tle=(line1, line2)` and `debris_tle=(line1, line2)` to `node.run_once()`
or `node.start()` to use current TLEs from Space-Track / CelesTrak instead of the
built-in defaults.
